# 🎙️ AURA Voice — F5-TTS Irish Voice Clone (Colab)\n\nThis notebook runs **F5-TTS** zero-shot voice cloning on a free T4 GPU.\nIt exposes a public API endpoint via Gradio that your AURA app can connect to.\n\n**F5-TTS** is faster and higher quality than XTTS-v2, and works on Python 3.12+.\n\n**Steps:**\n1. Run all cells (5 minutes total)\n2. Copy the public Gradio URL from Cell 5\n3. Paste it into your `.env` as `AURA_XTTS_URL`

In [ ]:
# Cell 1 — Install F5-TTS + dependencies (~2 min)\n!pip install -q f5-tts gradio scipy numpy

In [ ]:
# Cell 2 — Upload your Irish voice reference\nimport os\nfrom google.colab import files\n\nos.makedirs('voices', exist_ok=True)\n\n# Option A: Download from your public repo\nif not os.path.exists('voices/aura_irish.wav'):\n    print('Downloading Irish voice reference from GitHub...')\n    !wget -q -O voices/aura_irish.wav https://raw.githubusercontent.com/shelfgenius/permanentai-os/main/voices/aura_irish.wav\n    if os.path.exists('voices/aura_irish.wav') and os.path.getsize('voices/aura_irish.wav') > 1000:\n        print(f'Downloaded! Size: {os.path.getsize(\"voices/aura_irish.wav\"):,} bytes')\n    else:\n        print('Download failed. Upload manually:')\n        uploaded = files.upload()\n        for name, data in uploaded.items():\n            with open('voices/aura_irish.wav', 'wb') as f:\n                f.write(data)\n            print(f'Saved {name} as voices/aura_irish.wav')\nelse:\n    print(f'Voice reference already exists ({os.path.getsize(\"voices/aura_irish.wav\"):,} bytes)')

In [ ]:
# Cell 3 — Load F5-TTS model\nimport torch\nimport soundfile as sf\nfrom f5_tts.api import F5TTS\n\ndevice = 'cuda' if torch.cuda.is_available() else 'cpu'\nprint(f'Using device: {device}')\nif device == 'cuda':\n    print(f'GPU: {torch.cuda.get_device_name(0)}')\n    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')\n\nprint('Loading F5-TTS model (first run downloads ~1.2GB)...')\ntts = F5TTS(model_type='F5-TTS', device=device)\nprint('F5-TTS model loaded!')

In [ ]:
# Cell 4 — Quick test\nimport IPython.display as ipd\n\nwav, sr, _ = tts.infer(\n    ref_file='voices/aura_irish.wav',\n    ref_text='',  # leave empty for auto-transcription\n    gen_text='Hello! I am Aura, your AI assistant. How can I help you today?',\n)\n\nsf.write('test_output.wav', wav, sr)\nprint(f'Generated audio: {len(wav)/sr:.1f}s at {sr}Hz')\nipd.display(ipd.Audio('test_output.wav', autoplay=True))

In [ ]:
# Cell 5 — Start API server with public Gradio URL\nimport gradio as gr\nimport tempfile\nimport soundfile as sf\n\ndef synthesize(text, language='en'):\n    \"\"\"Generate speech cloning the Irish voice.\"\"\"\n    if not text or not text.strip():\n        return None\n    wav, sr, _ = tts.infer(\n        ref_file='voices/aura_irish.wav',\n        ref_text='',\n        gen_text=text.strip(),\n    )\n    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)\n    sf.write(tmp.name, wav, sr)\n    return tmp.name\n\ndef health():\n    return {'status': 'ready', 'device': device, 'model': 'f5-tts'}\n\nwith gr.Blocks(title='AURA Voice') as demo:\n    gr.Markdown('# AURA Voice — Irish Voice Clone (F5-TTS on Colab T4)')\n    text_in = gr.Textbox(label='Text', value='Hello, I am Aura.', lines=3)\n    btn = gr.Button('Speak', variant='primary')\n    audio_out = gr.Audio(label='Output', type='filepath')\n    btn.click(fn=synthesize, inputs=[text_in], outputs=audio_out, api_name='synthesize')\n    health_btn = gr.Button('Health Check', size='sm')\n    health_json = gr.JSON(label='Status')\n    health_btn.click(fn=health, outputs=health_json, api_name='health')\n\nprint('\\n' + '='*60)\nprint('  AURA Voice server starting...')\nprint('  Copy the PUBLIC URL below and set as AURA_XTTS_URL in .env')\nprint('='*60 + '\\n')\n\ndemo.launch(share=True, quiet=False)